# 02 — Experiments & Results

Load saved model predictions and metrics, reproduce comparison plots, tune thresholds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, confusion_matrix,
)

sns.set_theme(style='whitegrid', font_scale=1.1)

RESULTS_DIR = Path('../results')
FEATURES_DIR = Path('../data/features')

## 1. Load Predictions & Metrics

In [ ]:
def load_preds(model_name):
    d = np.load(RESULTS_DIR / f'{model_name}_predictions.npz')
    return d['y_true'], d['y_pred'], d['y_prob']

xgb_true, xgb_pred, xgb_prob = load_preds('xgboost')
mlp_true, mlp_pred, mlp_prob = load_preds('mlp')

with open(RESULTS_DIR / 'xgboost_metrics.json') as f:
    xgb_metrics = json.load(f)
with open(RESULTS_DIR / 'mlp_metrics.json') as f:
    mlp_metrics = json.load(f)

print('XGBoost test metrics:', xgb_metrics.get('test', {}))
print('MLP test metrics:', mlp_metrics.get('test', {}))

## 2. Comparison Table

In [ ]:
keys = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
        'precision_weighted', 'recall_weighted', 'f1_weighted', 'roc_auc']

xgb_t = xgb_metrics.get('test', {})
mlp_t = mlp_metrics.get('test', {})

comparison = pd.DataFrame({
    'XGBoost': [xgb_t.get(k, np.nan) for k in keys],
    'MLP': [mlp_t.get(k, np.nan) for k in keys],
}, index=keys)
comparison['Delta (MLP-XGB)'] = comparison['MLP'] - comparison['XGBoost']
comparison.round(4)

## 3. ROC & PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for (y_true, y_prob), name, color in [
    ((xgb_true, xgb_prob), 'XGBoost', '#2196F3'),
    ((mlp_true, mlp_prob), 'MLP', '#FF5722'),
]:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.4f})')

    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{name} (AP={ap:.4f})')

axes[0].plot([0,1],[0,1],'k--',lw=1); axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_pr_combined.png', dpi=150)
plt.show()

## 4. Threshold Analysis

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (y_true, y_prob), name, ax, color in [
    ((xgb_true, xgb_prob), 'XGBoost', axes[0], '#2196F3'),
    ((mlp_true, mlp_prob), 'MLP', axes[1], '#FF5722'),
]:
    f1s = [f1_score(y_true, (y_prob>=t).astype(int), average='macro', zero_division=0) for t in thresholds]
    accs = [accuracy_score(y_true, (y_prob>=t).astype(int)) for t in thresholds]
    ax.plot(thresholds, f1s, color='#4CAF50', label='F1 Macro')
    ax.plot(thresholds, accs, color=color, label='Accuracy')
    best_t = thresholds[np.argmax(f1s)]
    ax.axvline(best_t, color='red', linestyle='--', label=f'Best F1 threshold={best_t:.2f}')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{name} — Threshold Analysis')
    ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'threshold_analysis.png', dpi=150)
plt.show()

## 5. Feature Importance Comparison

In [ ]:
with open(RESULTS_DIR / 'xgboost_feature_importance.json') as f:
    xgb_imp = json.load(f)

# Sort and display top 15
top15 = sorted(xgb_imp.items(), key=lambda x: x[1], reverse=True)[:15]
names, vals = zip(*top15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(15), list(vals)[::-1], color='#2196F3', alpha=0.8)
ax.set_yticks(range(15))
ax.set_yticklabels(list(names)[::-1])
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost Top 15 Feature Importances')
plt.tight_layout()
plt.show()

## 6. MLP Training History

In [ ]:
history = mlp_metrics.get('training_history', [])
if history:
    epochs = [h['epoch'] for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss = [h['val_loss'] for h in history]
    val_auc = [h['val_auc'] for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(epochs, train_loss, label='Train Loss', color='#4CAF50')
    ax1.plot(epochs, val_loss, label='Val Loss', color='#FF5722')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
    ax1.set_title('MLP Training Loss'); ax1.legend()

    ax2.plot(epochs, val_auc, color='#2196F3', marker='o')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val ROC-AUC')
    ax2.set_title('MLP Val AUC')
    plt.tight_layout()
    plt.show()
else:
    print('No training history found. Run 04_mlp_model.py first.')